In [19]:
import matplotlib.pyplot as plt

plt.style.use('default')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['savefig.edgecolor'] = 'white'

plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['axes.titlecolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'

plt.rcParams['axes.edgecolor'] = 'black'

plt.rcParams['savefig.transparent'] = False

In [20]:
def apply_white_theme(ax):
    ax.set_facecolor('white')

    ax.title.set_color('black')
    ax.xaxis.label.set_color('black')
    ax.yaxis.label.set_color('black')
    ax.tick_params(axis='x', colors='black')
    ax.tick_params(axis='y', colors='black')

    for spine in ax.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.0)

    ax.grid(True, linestyle='--', alpha=0.4, color='gray')

In [21]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_FILE = 'Food_Inspections_Cleaned.csv'
OUTPUT_DIR = 'output_visualizations'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

df = pd.read_csv(INPUT_FILE, low_memory=False)

text_cols = ['DBA Name', 'AKA Name', 'Address', 'Facility Type', 'Risk', 'Results', 'Inspection Type']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()

if 'Zip' in df.columns:
    df['Zip'] = pd.to_numeric(df['Zip'], errors='coerce')

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]:,} cols")

Loaded: 264,709 rows × 17 cols


In [22]:
def plot_missing_value_percentage(df, output_dir='output_visualizations'):
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8), facecolor='white')
    ax.barh(missing_pct.index, missing_pct.values)

    ax.set_xlabel('Missing Percentage (%)', color='black')
    ax.set_ylabel('Columns', color='black')
    ax.set_title('Missing Value Percentage by Column', color='black')

    apply_white_theme(ax)
    ax.grid(axis='x', linestyle='--', alpha=0.5, color='gray')

    output_path = os.path.join(output_dir, 'missing_value_percentage.png')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='white', transparent=False)
    plt.close()

    print(f"Saved: {output_path}")
    return missing_pct

In [23]:
def plot_category_distribution(df, column, top_n=15, output_dir='output_visualizations'):
    if column not in df.columns:
        print(f"Column not found: {column}")
        return

    counts = df[column].fillna('MISSING').value_counts().head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')
    ax.bar(counts.index, counts.values)

    ax.set_xlabel(column, color='black')
    ax.set_ylabel('Count', color='black')
    ax.set_title(f'Top {top_n} Categories in {column}', color='black')

    apply_white_theme(ax)
    ax.grid(axis='y', linestyle='--', alpha=0.5, color='gray')
    plt.xticks(rotation=45, ha='right')

    safe_name = column.lower().replace(' ', '_').replace('#', 'num')
    output_path = os.path.join(output_dir, f'category_distribution_{safe_name}.png')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='white', transparent=False)
    plt.close()

    print(f"Saved: {output_path}")
    return counts

results_counts = plot_category_distribution(df, 'Results', top_n=10, output_dir=OUTPUT_DIR)
risk_counts = plot_category_distribution(df, 'Risk', top_n=10, output_dir=OUTPUT_DIR)
facility_counts = plot_category_distribution(df, 'Facility Type', top_n=15, output_dir=OUTPUT_DIR)

Saved: output_visualizations\category_distribution_results.png
Saved: output_visualizations\category_distribution_risk.png
Saved: output_visualizations\category_distribution_facility_type.png


In [24]:
def compute_fd_confidence(df, lhs, rhs):
    group_nunique = df.groupby(lhs)[rhs].nunique(dropna=True)
    total_groups = len(group_nunique)
    violating_groups = (group_nunique > 1).sum()

    if total_groups == 0:
        return 0.0

    confidence = 1 - violating_groups / total_groups
    return round(confidence, 4)

def plot_fd_confidence_ranking(df, fd_list, output_dir='output_visualizations'):
    rows = []
    for lhs, rhs in fd_list:
        if lhs not in df.columns or rhs not in df.columns:
            continue
        conf = compute_fd_confidence(df, lhs, rhs)
        rows.append({
            'FD': f'{lhs} -> {rhs}',
            'Confidence': conf
        })

    fd_conf_df = pd.DataFrame(rows).sort_values(by='Confidence', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')
    ax.bar(fd_conf_df['FD'], fd_conf_df['Confidence'])

    ax.set_ylabel('FD Confidence', color='black')
    ax.set_xlabel('Functional Dependency', color='black')
    ax.set_title('Functional Dependency Confidence Ranking', color='black')
    ax.set_ylim(0.85, 1.01)

    apply_white_theme(ax)
    ax.grid(axis='y', linestyle='--', alpha=0.5, color='gray')
    plt.xticks(rotation=30, ha='right')

    output_path = os.path.join(output_dir, 'fd_confidence_ranking.png')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='white', transparent=False)
    plt.close()

    print(f"Saved: {output_path}")
    return fd_conf_df

fd_list = [
    ('Address', 'Zip'),
    ('License #', 'Address'),
    ('License #', 'Zip'),
    ('License #', 'Risk'),
    ('License #', 'Facility Type')
]

fd_conf_df = plot_fd_confidence_ranking(df, fd_list, OUTPUT_DIR)
print(fd_conf_df)

Saved: output_visualizations\fd_confidence_ranking.png
                           FD  Confidence
0              Address -> Zip      0.9980
2            License # -> Zip      0.9979
3           License # -> Risk      0.9946
1        License # -> Address      0.9934
4  License # -> Facility Type      0.9923


In [25]:
def compute_fd_violation_counts(df, lhs, rhs):
    group_nunique = df.groupby(lhs)[rhs].nunique(dropna=True)
    violating_groups = (group_nunique > 1).sum()
    total_groups = len(group_nunique)
    return violating_groups, total_groups

def plot_fd_violation_counts(df, fd_list, output_dir='output_visualizations'):
    rows = []
    for lhs, rhs in fd_list:
        if lhs not in df.columns or rhs not in df.columns:
            continue
        violating_groups, total_groups = compute_fd_violation_counts(df, lhs, rhs)
        rows.append({
            'FD': f'{lhs} -> {rhs}',
            'Violating Groups': violating_groups,
            'Total Groups': total_groups
        })

    fd_violation_df = pd.DataFrame(rows).sort_values(by='Violating Groups', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')
    ax.bar(fd_violation_df['FD'], fd_violation_df['Violating Groups'])

    ax.set_ylabel('Number of Violating Groups', color='black')
    ax.set_xlabel('Functional Dependency', color='black')
    ax.set_title('FD Violation Counts', color='black')

    apply_white_theme(ax)
    ax.grid(axis='y', linestyle='--', alpha=0.5, color='gray')
    plt.xticks(rotation=30, ha='right')

    output_path = os.path.join(output_dir, 'fd_violation_counts.png')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='white', transparent=False)
    plt.close()

    print(f"Saved: {output_path}")
    return fd_violation_df

fd_violation_df = plot_fd_violation_counts(df, fd_list, OUTPUT_DIR)
print(fd_violation_df)

Saved: output_visualizations\fd_violation_counts.png
                           FD  Violating Groups  Total Groups
4  License # -> Facility Type               341         44044
1        License # -> Address               291         44044
3           License # -> Risk               240         44044
2            License # -> Zip                92         44044
0              Address -> Zip                38         19074


In [26]:
def main():
    df = pd.read_csv(INPUT_FILE, low_memory=False)

    text_cols = ['DBA Name', 'AKA Name', 'Address', 'Facility Type', 'Risk', 'Results', 'Inspection Type']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()

    if 'Zip' in df.columns:
        df['Zip'] = pd.to_numeric(df['Zip'], errors='coerce')

    plot_missing_value_percentage(df, OUTPUT_DIR)
    plot_category_distribution(df, 'Results', top_n=10, output_dir=OUTPUT_DIR)
    plot_category_distribution(df, 'Risk', top_n=10, output_dir=OUTPUT_DIR)
    plot_category_distribution(df, 'Facility Type', top_n=15, output_dir=OUTPUT_DIR)

    fd_list = [
        ('Address', 'Zip'),
        ('License #', 'Address'),
        ('License #', 'Zip'),
        ('License #', 'Risk'),
        ('License #', 'Facility Type')
    ]

    plot_fd_confidence_ranking(df, fd_list, OUTPUT_DIR)
    plot_fd_violation_counts(df, fd_list, OUTPUT_DIR)

    print("\nAll required charts have been generated successfully.")

if __name__ == '__main__':
    main()

Saved: output_visualizations\missing_value_percentage.png
Saved: output_visualizations\category_distribution_results.png
Saved: output_visualizations\category_distribution_risk.png
Saved: output_visualizations\category_distribution_facility_type.png
Saved: output_visualizations\fd_confidence_ranking.png
Saved: output_visualizations\fd_violation_counts.png

All required charts have been generated successfully.
